In [2]:
import torch.nn as nn 
import torchvision
import numpy as np
import torch

```
class:
    def __init__(self, in, out, numclass, downsample ...):
        self ....
        def forward
            return
```

In [3]:
class SimpleCNNs(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.num_class = num_class # attribute
        # |> methods
        self.feature = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.adap_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(256, num_class)
    def forward(self, x):
        x = self.feature(x)
        x = self.adap_pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

model = SimpleCNNs(num_class=5)
dummy_input = torch.randn(3, 3, 224, 224)
output = model(dummy_input)
print(f'output: {output}')

output: tensor([[-0.4652, -0.5913,  0.6873,  0.5793, -0.0551],
        [-0.4418, -0.6279,  0.7173,  0.5753, -0.0211],
        [-0.4315, -0.6115,  0.7373,  0.5223, -0.0349]],
       grad_fn=<AddmmBackward0>)


In [4]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

        self.features = nn.Sequential(
            nn.Conv2d(self.in_channels, self.out_channels, kernel_size=3, stride=self.stride, padding=1, bias=False),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(),

            nn.Conv2d(self.out_channels, self.out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(self.out_channels)
        )

        self.downsample = None
        if self.stride != 1 or self.in_channels != self.out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, self.out_channels, kernel_size=1, stride=self.stride, bias=False),
                nn.BatchNorm2d(self.out_channels)
            )

    def forward(self, x):
        identity = x
        out = self.features(x)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        out = torch.relu(out)
        return out

In [5]:
class Resnet18(nn.Module):
    def __init__(self, num_class=1000, in_channels=64):
        super().__init__()
        self.num_class = num_class
        self.in_channels = in_channels

        self.feature = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        
        self.layer_1 = self._make_layers(64, 2, 1)
        self.layer_2 = self._make_layers(128, 2, 2)
        self.layer_3 = self._make_layers(256, 2, 2)
        self.layer_4 = self._make_layers(512, 2, 2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, self.num_class)

    def _make_layers(self, output_channels, blocks, stride=1):
        layers = []
        layers.append(BasicBlock(self.in_channels, output_channels, stride))
        self.in_channels = output_channels
        for i in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, output_channels, stride=1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.feature(x)
        x = self.layer_1(x)
        x = self.layer_2(x)
        x = self.layer_3(x)
        x = self.layer_4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, dim = 1)
        x = self.fc(x)

        return x
            

In [6]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growrate = 32):
        super().__init__()
        self.in_channels = in_channels
        self.growrate = growrate
        self.feature = nn.Sequential(
            nn.BatchNorm2d(self.in_channels),
            nn.ReLU(),
            nn.Conv2d(self.in_channels, 4*self.growrate, kernel_size=1, bias=False),
            nn.BatchNorm2d(4*self.growrate),
            nn.ReLU(),
            nn.Conv2d(4*growrate, growrate, kernel_size=3, padding=1, bias=False)
        )
    def forward(self, init_feat):
        if isinstance(init_feat, torch.Tensor):
            init_feat = [init_feat]
        x = torch.concat(init_feat, dim = 1)
        new_feat = self.feature(x)
        return new_feat
class DenseBlock(nn.Module):
    def __init__(self, num_layer, in_channels, growrate):
        super().__init__()
        for i in range(num_layer):
            layer = DenseLayer(in_channels + i*growrate, growrate)
            self.add_module(f'dense_layer: {i+1}', layer)
    def forward(self, init_feat):
        feats = [init_feat]
        for name, layer in self.named_children():
            new_feat = layer(feats)
            feats.append(new_feat)
        return torch.cat(feats, dim = 1)
class Transition(nn.Module):
   def __init__(self, in_channel, out_channel):
      super(Transition, self).__init__()
      self.bn = nn.BatchNorm2d(in_channel)
      self.relu = nn.ReLU()
      self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=1, stride=1, bias=False)
      self.avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
   def forward(self, x):
      out = self.bn(x)
      out = self.relu(out)
      out = self.conv1(out)
      out = self.avg_pool(out)

      return out

In [7]:
class DenseNet121(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        self.block1 = DenseBlock(num_layer=6, in_channels=64, growrate=32)
        self.trans1 = Transition(in_channel=256, out_channel=128)
        
        self.block2 = DenseBlock(num_layer=12, in_channels=128, growrate=32)
        self.trans2 = Transition(in_channel=512, out_channel=256)
        
        self.block3 = DenseBlock(num_layer=24, in_channels=256, growrate=32)

        self.final_bn = nn.BatchNorm2d(1024)
        self.final_relu = nn.ReLU()
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(1024, num_class)
    def forward(self, x):
        # Đi qua tầng xử lý ảnh gốc ban đầu
        x = self.feature(x)
        
        # Khối 1 + Hạ kích thước
        x = self.block1(x)
        x = self.trans1(x)
        
        # Khối 2 + Hạ kích thước
        x = self.block2(x)
        x = self.trans2(x)
        
        # Khối cuối cùng (Giữ nguyên kích thước sắc nét của đặc trưng)
        x = self.block3(x)
        
        # Đi vào tầng phân loại
        x = self.final_bn(x)
        x = self.final_relu(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1) # Giữ nguyên chiều Batch_size như bạn đã hiểu!
        x = self.fc(x)
        
        return x

In [8]:
class SqueezeExcite(nn.Module):
    def __init__(self, ch, se_ch):
        super().__init__()
        self.scale = nn.AdaptiveAvgPool2d(1)
        self.features = nn.Sequential(
            nn.Conv2d(ch, se_ch, kernel_size=1, bias=False),
            nn.SiLU(),
            nn.Conv2d(se_ch, ch, kernel_size=1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        scale = self.scale(x)
        scale = self.features(scale)
        return x * scale
    
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expand, kernel, stride):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.expand = expand
        self.stride = stride
        self.hidden_channels = in_channels * expand

        self.use_expand = self.expand != 1
        if self.use_expand:
            self.conv1 = nn.Conv2d(in_channels, self.hidden_channels, kernel_size=1, stride=1, padding=0, bias=False)
            self.bn1 = nn.BatchNorm2d(self.hidden_channels)
            self.act1 = nn.SiLU()

        padding = kernel // 2
        self.depthwise = nn.Conv2d(
            self.hidden_channels, self.hidden_channels, groups=self.hidden_channels, 
            kernel_size=kernel, stride=stride, padding=padding, bias=False
        )
        self.bn2 = nn.BatchNorm2d(self.hidden_channels)
        self.act2 = nn.SiLU()

        self.se_ch = max(1, self.hidden_channels // 4)
        self.se = SqueezeExcite(self.hidden_channels, se_ch=self.se_ch)
        
        self.project = nn.Conv2d(self.hidden_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        
        self.use_residual = self.stride == 1 and self.in_channels == self.out_channels

    def forward(self, x):
        identity = x 
        
        if self.use_expand:
            out = self.act1(self.bn1(self.conv1(x)))
        else:
            out = x
            
        out = self.act2(self.bn2(self.depthwise(out)))
        out = self.se(out)
        
        out = self.bn3(self.project(out))

        if self.use_residual:
            out = out + identity
            
        return out

In [9]:
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        
        # 1. Tầng thô ban đầu (Stem Layer): Nhận ảnh 3 kênh RGB -> Nhả ra 32 kênh
        # Hạ kích thước ảnh xuống một nửa bằng stride=2
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )
        
        # 2. CẤU HÌNH CÁC STAGES CỦA EFFICIENTNET-B0
        # Mẹo: Khối đầu tiên của mỗi Stage có thể nhận stride=2 để hạ size ảnh, 
        # các khối lặp lại phía sau bắt buộc dùng stride=1 để giữ nguyên kích thước.
        
        # Stage 1: Vào 32, Ra 16 | Hệ số expand=1, Kernel=3x3, Stride=1 (Không lặp lại)
        self.stage1 = MBConv(32, 16, expand=1, kernel=3, stride=1)
        
        # Stage 2: Vào 16, Ra 24 | Hệ số expand=6, Kernel=3x3, Stride=2 (Lặp lại 2 lần)
        self.stage2 = nn.Sequential(
            MBConv(16, 24, expand=6, kernel=3, stride=2), # Khối chuyển giao (stride=2)
            MBConv(24, 24, expand=6, kernel=3, stride=1)  # Khối lặp lại (stride=1)
        )
        
        # Stage 3: Vào 24, Ra 40 | Hệ số expand=6, Kernel=5x5, Stride=2 (Lặp lại 2 lần)
        self.stage3 = nn.Sequential(
            MBConv(24, 40, expand=6, kernel=5, stride=2),
            MBConv(40, 40, expand=6, kernel=5, stride=1)
        )
        
        # Stage 4: Vào 40, Ra 80 | Hệ số expand=6, Kernel=3x3, Stride=2 (Lặp lại 3 lần)
        self.stage4 = nn.Sequential(
            MBConv(40, 80, expand=6, kernel=3, stride=2),
            MBConv(80, 80, expand=6, kernel=3, stride=1),
            MBConv(80, 80, expand=6, kernel=3, stride=1)
        )
        
        # Stage 5: Vào 80, Ra 112 | Hệ số expand=6, Kernel=5x5, Stride=1 (Lặp lại 3 lần)
        self.stage5 = nn.Sequential(
            MBConv(80,  112, expand=6, kernel=5, stride=1),
            MBConv(112, 112, expand=6, kernel=5, stride=1),
            MBConv(112, 112, expand=6, kernel=5, stride=1)
        )
        
        # Stage 6: Vào 112, Ra 192 | Hệ số expand=6, Kernel=5x5, Stride=2 (Lặp lại 4 lần)
        self.stage6 = nn.Sequential(
            MBConv(112, 192, expand=6, kernel=5, stride=2),
            MBConv(192, 192, expand=6, kernel=5, stride=1),
            MBConv(192, 192, expand=6, kernel=5, stride=1),
            MBConv(192, 192, expand=6, kernel=5, stride=1)
        )
        
        # Stage 7: Vào 192, Ra 320 | Hệ số expand=6, Kernel=3x3, Stride=1 (Không lặp lại)
        self.stage7 = MBConv(192, 320, expand=6, kernel=3, stride=1)
        
        # 3. TẦNG ĐẦU RA KẾT THÚC MẠNG (Head Layer)
        # Mở rộng kênh lên 1280 bằng Conv 1x1 trước khi nén phẳng dữ liệu
        self.conv_head = nn.Sequential(
            nn.Conv2d(320, 1280, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU()
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1) # Ép ảnh về 1x1
        self.fc = nn.Linear(1280, num_classes)     # Tầng phân loại tuyến tính

    def forward(self, x):
        # Chạy qua tầng thô
        x = self.stem(x)
        
        # Đi qua chuỗi các Stage đặc trưng
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)
        x = self.stage6(x)
        x = self.stage7(x)
        
        # Đi vào tầng phân loại đầu ra
        x = self.conv_head(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1) # Giữ lại chiều Batch_size thần thánh!
        x = self.fc(x)
        return x